# Linear Regression Deep Dive

# Load packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_breuschpagan
from sklearn.preprocessing import PowerTransformer, QuantileTransformer, StandardScaler

# Motivating Dataset 1: CERN

Research question: For particles with very little mass, can we predict their energy given momentum alone?

## Load in the Data

In [ ]:
cern = pd.read_csv("../Unit 3/data/cern.csv")

## Wrangle the Data

1.    Subset to only include `Run` 165617

2.    Select variables of interest: `E1`, `px1`, `py1`, `pz1`

3.    Name final object as `cern_voi` (variables of interest)

In [ ]:
cern_voi = cern.loc[cern["Run"].eq(165617), ["E1", "px1", "py1", "pz1"]].copy()
cern_voi.head()

## Visualize Dataset using `ggpairs`

In [ ]:
sns.pairplot(cern_voi, diag_kind="hist")
plt.show()

## Simple Linear Regression

In [ ]:
smf.ols("E1 ~ px1", data=cern_voi).fit().params

<!--- OR ---->

In [ ]:
sm.OLS(cern_voi["E1"], sm.add_constant(cern_voi["px1"])).fit().params

<!--- not super informative, though... --->

In [ ]:
print(smf.ols("E1 ~ px1", data=cern_voi).fit().summary())

## Your Turn

> Perform simple linear regression using the `py1` as the predictor of `E1` (print out the summary)

In [ ]:
py1_model = smf.ols("E1 ~ py1", data=cern_voi).fit()
print(py1_model.summary())

## Plotting

In [ ]:
sns.regplot(data=cern_voi, x="px1", y="E1", ci=95)
plt.xlabel("Momentum in the x-direction")
plt.ylabel("Energy")
plt.show()

Doesn't seem to be working well...

Let's try using 3D momentum

## Your Turn: Create 3D Momentum

> Compute 3D momentum by taking the square root of the sum of the squares of these three variables: `px1`, `py1`, `pz1`

1.    Square the variables

2.    Sum the squared values

3.    Take the square root of the sum

This computation *can* be done in a single line of code!

*Hint*: you'll need to sum the rows!

> Store the values as `p1` in your `cern_voi` data frame

In [ ]:
cern_voi["p1"] = np.sqrt((cern_voi[["px1", "py1", "pz1"]] ** 2).sum(axis=1))
cern_voi.head()

> Now, perform simple linear regression

In [ ]:
p1_model = smf.ols("E1 ~ p1", data=cern_voi).fit()
print(p1_model.summary())

> Finally, plot

In [ ]:
sns.regplot(data=cern_voi, x="p1", y="E1", ci=95)
plt.xlabel("3D Momentum")
plt.ylabel("Energy")
plt.show()

> Reflect: Why would you expect such different results based on one direction of momentum (`px1` or x-direction) versus three-dimensional momentum (`p1`)?

**Solution:** Energy depends on the magnitude of the particle momentum, not on a single directional component. A particle can have substantial momentum in the y- or z-direction even when `px1` is small, so `px1` alone discards relevant information. The 3D magnitude `p1` combines all three components and should therefore track energy much more closely.




<!--- Back to slides --->

## Assumptions and Diagnostics

Let's plot all three directions of `p`

In [ ]:
long = cern_voi.melt(id_vars="E1", value_vars=["px1","py1","pz1"], var_name="Direction", value_name="Magnitude")
sns.lmplot(data=long, x="Magnitude", y="E1", hue="Direction", height=5, aspect=1.3)
plt.show()

Now, let's make a linear model of momentum to predict energy

In [ ]:
E1_model = smf.ols("E1 ~ px1 + py1 + pz1", data=cern_voi).fit()
print(E1_model.summary())

To plot, we need to get our predictions

In [ ]:
cern_voi["E1_hat"] = E1_model.predict(cern_voi)
sns.regplot(data=cern_voi, x="E1_hat", y="E1", ci=95)
plt.xlabel("Predicted Energy")
plt.ylabel("Observed Energy")
plt.show()

> What do you notice about this plot (e.g., do our predictions seem to fit)?

**Solution:** The observed-versus-predicted plot should be inspected against the 45-degree ideal. The predictions capture some systematic variation in energy, but using the three directional components as separate linear predictors is not equivalent to using the nonlinear momentum magnitude $\sqrt{p_x^2+p_y^2+p_z^2}$, so departures from a perfect fit are expected.




<!--- Back to slides --->

### Multicollinearity

In [ ]:
X = pd.DataFrame(E1_model.model.exog, columns=E1_model.model.exog_names)
pd.DataFrame({"Feature":X.columns, "VIF":[variance_inflation_factor(X.values,i) for i in range(X.shape[1])]})

### Outliers

To detect outliers, we'll use three common and related metrics: difference-in-fits (or `dffits`), Cook's distance, and difference-in-betas (or `dfbetas`)

Compute statistics

In [ ]:
E1_influence = E1_model.get_influence()
influence_df = E1_influence.summary_frame()
influence_df.head()

#### difference-in-fits

Compute statistic

In [ ]:
vals = E1_influence.dffits[0]
plt.vlines(np.arange(len(vals)), 0, vals)
plt.axhline(0, color="black", linewidth=0.8)
plt.show()

A general guideline is:

$$
|DFFITS_i| > 2 \cdot \sqrt{\frac{p + 1}{n}},
$$

where $p$ is the number of predictors and $n$ is the sample size.

> Compute the above equation and use `which()` to identify outlying indices

In [ ]:
n=len(cern_voi); p=3
dffits_cutoff=2*np.sqrt((p+1)/n)
print(dffits_cutoff)
np.where(np.abs(E1_influence.dffits[0]) > dffits_cutoff)[0]

> Do there appear to be any outliers?

**Solution:** Use the indices printed by the preceding cutoff calculation. Any listed cases exceed the stated influence cutoff and should be investigated; they should not be deleted automatically.



#### Cook's distance

Compute statistic

In [ ]:
cooks_d = E1_influence.cooks_distance[0]
plt.vlines(np.arange(len(cooks_d)), 0, cooks_d)
plt.show()

A general guideline is:

$$
D > \frac{4}{n - k - 1}
$$

> Compute the above equation and use `which()` to identify outlying indices

In [ ]:
n=len(cern_voi); k=3
cooks_cutoff=4/(n-k-1)
print(cooks_cutoff)
np.where(cooks_d > cooks_cutoff)[0]

> Do there appear to be any outliers?

**Solution:** Use the indices printed by the preceding cutoff calculation. Any listed cases exceed the stated influence cutoff and should be investigated; they should not be deleted automatically.

Fewer outliers than DFFIT but still many outliers.



#### difference-in-betas

Compute statistic

In [ ]:
dfbetas = E1_influence.dfbetas
plt.vlines(np.arange(len(dfbetas)), 0, dfbetas[:,0])
plt.show()

A general guideline is:

$$
|dfbetas| > \frac{2}{\sqrt{n}},
$$

where $p$ is the number of predictors and $n$ is the sample size.

> Compute the above equation and use `which()` to identify outlying indices

In [ ]:
dfbetas_cutoff=2/np.sqrt(len(cern_voi))
print(dfbetas_cutoff)
np.where(np.abs(dfbetas[:,0]) > dfbetas_cutoff)[0]

> Was there a consensus across the influence measures? Do there appear to be any outliers?

**Solution:** Compare the cases flagged by DFFITS, Cook's distance, and DFBETAS. Cases flagged by multiple measures deserve the most attention because the diagnostics agree that they have unusual influence on the fitted model. A flagged case is a prompt for investigation, not automatic removal.



### Normal Residuals

<!---

Shapiro-Wilks test of normality has the following hypotheses

Null (H_0): The values are not significantly different from normal

Alternative (H_A): The values are significantly different from normal

--->

> Before running this code, what do you expect to happen?

A.    Retain the null: The residuals are not significantly different from normal

B.    Reject the null: The residuals are significantly different from normal

In [ ]:
print(stats.shapiro(E1_model.resid))
sns.scatterplot(x=E1_model.fittedvalues, y=E1_model.resid)
plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("Fitted values"); plt.ylabel("Residuals"); plt.show()

### Homoscedasticity

In [ ]:
sm.qqplot(E1_model.resid, line="45")
plt.show()

## Resolutions

We'll start and work backwards on the resolutions for the linear model assumptions.

### Homoscedasticity

If homoscedasticity is an issue, then we can estimate corrected standard errors using the \{sandwich\} package

In [ ]:
print(E1_model.get_robustcov_results(cov_type="HC1").summary())

> Did anything change?

**Solution:** The coefficient estimates remain the same because the fitted OLS line has not changed. The heteroscedasticity-consistent covariance estimate changes the standard errors and therefore can change the test statistics, p-values, and confidence intervals.



We'll also check out whether there is independence using the Durbin-Watson test in the \{car\} package

In [ ]:
print("Durbin-Watson:", durbin_watson(E1_model.resid))

Here's how to interpret the test:

$< 2$ = positive autocorrelation
$2$ = no autocorrelation
$> 2$ = negative autocorrelation

## Your Turn: Predicting Life Expectancy

Your goal is to work through a similar pipeline as we just walked through above.

Our objective will be to build a model to predict life expectancy.

> Let's start by loading in the dataset: `life_expectancy.csv`

In [ ]:
life = pd.read_csv("life_expectancy.csv")
print(life.shape)
life.columns.tolist()

Here's our variables of interest:

`Life.expectancy` (numeric; 48.1-89.0) = average life expectancy in years

`Alcohol` (numeric; 0.01-15.19) = per capita consumption in liters of pure alcohol

`Schooling` (numeric; 4.9-20.4) = average number of years in school
 
`percentage.expenditure` (numeric; 0-100) = expenditure on health as a percentage of GDP per capita

`GDP` (numeric) = gross domestic product in millions (USD)

> Select these variables and rename them as you see fit

In [ ]:
life_voi = life[["Life expectancy","Alcohol","Schooling","percentage expenditure","GDP"]].copy()
life_voi.columns=["life_expectancy","alcohol","schooling","health_expenditure","gdp"]
life_voi=life_voi.dropna()
life_voi.info()

> Run `ggpairs()` to get an idea of the relationships and distributions

In [ ]:
sns.pairplot(life_voi, diag_kind="hist")
plt.show()

> Reflect on the distributions -- do all variables appear to be normal?

**Solution:** No. In the supplied life-expectancy data, GDP and percentage health expenditure are strongly right-skewed, while the other variables are closer to symmetric but are not all normally distributed. Predictor normality is not itself an OLS requirement; the plots motivate later checks and possible transformations.



We'll come back to these distributions later. For now, let's try to run our model

> Perform linear regression and print the summary

In [ ]:
life_model=smf.ols("life_expectancy ~ alcohol + schooling + health_expenditure + gdp",data=life_voi).fit()
print(life_model.summary())

### Check for assumptions (follow workflow above)

> Multicollinearity?

In [ ]:
X=pd.DataFrame(life_model.model.exog,columns=life_model.model.exog_names)
pd.DataFrame({"Feature":X.columns,"VIF":[variance_inflation_factor(X.values,i) for i in range(X.shape[1])]})

> Any multicollinearity? If so, remove a variable and re-run the model

In [ ]:
# GDP and health_expenditure are strongly collinear; retain GDP and remove health_expenditure.
life_model2=smf.ols("life_expectancy ~ alcohol + schooling + gdp",data=life_voi).fit()
print(life_model2.summary())
X2=pd.DataFrame(life_model2.model.exog,columns=life_model2.model.exog_names)
pd.DataFrame({"Feature":X2.columns,"VIF":[variance_inflation_factor(X2.values,i) for i in range(X2.shape[1])]})

> Did the results change?

**Solution:** Yes, but the substantive conclusion is similar: schooling remains the dominant predictor. GDP and percentage health expenditure are strongly redundant (their correlation is about 0.91 and their initial VIFs are about 6), so retaining one removes that source of multicollinearity and changes the conditional coefficient estimates and their standard errors.




> Outliers?

In [ ]:
life_inf=life_model2.get_influence()
n=len(life_voi); k=3
dffits_cut=2*np.sqrt((k+1)/n); cooks_cut=4/(n-k-1); dfb_cut=2/np.sqrt(n)
idx_dffits=np.where(np.abs(life_inf.dffits[0])>dffits_cut)[0]
idx_cooks=np.where(life_inf.cooks_distance[0]>cooks_cut)[0]
idx_dfbetas=np.where(np.max(np.abs(life_inf.dfbetas),axis=1)>dfb_cut)[0]
outlier_pos=np.unique(np.concatenate([idx_dffits,idx_cooks,idx_dfbetas]))
outlier_pos

> Any outliers? If so, remove observation(s) and re-run the model

In [ ]:
life_clean=life_voi.drop(life_voi.index[outlier_pos]).copy()
life_model3=smf.ols("life_expectancy ~ alcohol + schooling + gdp",data=life_clean).fit()
print(life_model3.summary())

> Normal residuals?

In [ ]:
print(stats.shapiro(life_model3.resid))
sns.scatterplot(x=life_model3.fittedvalues,y=life_model3.resid)
plt.axhline(0,color="black",linewidth=.8)
plt.xlabel("Fitted values"); plt.ylabel("Residuals"); plt.show()

> Are the residuals normal?

**Solution:** For the initial complete-case model, Shapiro--Wilk gives approximately $p=0.005$, providing evidence against normally distributed residuals. After model changes/outlier handling, use the printed test together with the residual and Q--Q plots; a non-significant test alone does not prove normality.



> Homoscedasticity?

In [ ]:
sm.qqplot(life_model3.resid,line="45")
plt.show()

Before jumping to outliers, let's revisit the distributions (take another look at the `ggpairs` plot above)

> Let's transform our variables to be more normal

<!--- Hint: one of these variables should have been removed due to multicollinearity --->

We'll use the \{bestNormalize\} package to search over many different transformations to identify the one that gets our data closest to normal. Here's a list of transformations it applies:

arcsinh:

$$
\sinh^{-1}(x)
$$

box-cox:

$$
\begin{cases}
\frac{(y + \lambda_2)^{\lambda_1 - 1}}{\lambda_1}, \text{if} \ \lambda_1 \neq 0 \\
\log(y + \lambda_2), \text{if} \lambda_1 = 0
\end{cases}
$$

center and scale: `scale()`

$$
\frac{x - \bar{x}}{s_x}
$$

double reversed log:

$$
-\log_b(x + a)
$$

exponent:

$$
e^x
$$

log:

$$
\log_b(x + a)
$$

orderNorm:

$$
\Phi^{-1} \left( \frac{\text{rank}(x)}{n + 1} \right)
$$

where $\Phi^{-1}$ is the inverse cumulative normal distribution function (`qnorm()`)

square root:

$$
\sqrt{x + a}
$$

yeo-johnson: too complex to precisely display as in-line text

> Create an object called `life_voi` with the data frame you've been using

In [ ]:
life_voi = life_clean[["life_expectancy","alcohol","schooling","gdp"]].copy()

This code is provided for you:

In [ ]:
# Python equivalent: choose a normalizing transform for each variable from Yeo-Johnson and rank-based inverse normal.
from scipy.stats import shapiro, norm, rankdata

def best_normalize_series(s):
    x=s.to_numpy().reshape(-1,1)
    yj=PowerTransformer(method="yeo-johnson", standardize=True).fit_transform(x).ravel()
    rn=norm.ppf(rankdata(s)/(len(s)+1))
    z=StandardScaler().fit_transform(x).ravel()
    candidates={"yeo-johnson":yj,"orderNorm":rn,"center-and-scale":z}
    scores={name:shapiro(vals).statistic for name,vals in candidates.items()}
    choice=max(scores,key=scores.get)
    return candidates[choice], choice

life_transformed=pd.DataFrame(index=life_voi.index)
life_transformations={}
for col in life_voi.columns:
    life_transformed[col], life_transformations[col]=best_normalize_series(life_voi[col])
life_transformed.head()

Here's how to access which transformations were used (if any):

In [ ]:
life_transformations

> Re-fit your linear model using the `life_transformed` object

In [ ]:
life_transformed_model=smf.ols("life_expectancy ~ alcohol + schooling + gdp",data=life_transformed).fit()
print(life_transformed_model.summary())

> Did your model prediction ($R^2$) get worse or better?

**Solution:** Compare the printed $R^2$ with the previous model on the corresponding analysis sample. Transformation can improve distributional behavior without necessarily increasing $R^2$ on the original response scale, so the direction should be reported from the fitted output rather than assumed.



> Interpret the `schooling` parameter. (hint: the values have been standardized!)

**Solution:** Because the transformed/standardized variables are on standardized scales, the schooling coefficient represents the expected change in transformed life expectancy, in standard-deviation units, associated with a one-standard-deviation increase in transformed schooling, holding the other predictors constant.



> Normal residuals?

In [ ]:
print(stats.shapiro(life_transformed_model.resid))
sns.scatterplot(x=life_transformed_model.fittedvalues,y=life_transformed_model.resid)
plt.axhline(0,color="black",linewidth=.8)
plt.xlabel("Fitted values"); plt.ylabel("Residuals"); plt.show()

> Are the residuals normal?

**Solution:** For the initial complete-case model, Shapiro--Wilk gives approximately $p=0.005$, providing evidence against normally distributed residuals. After model changes/outlier handling, use the printed test together with the residual and Q--Q plots; a non-significant test alone does not prove normality.



> Homoscedasticity?

In [ ]:
sm.qqplot(life_transformed_model.resid,line="45")
plt.show()